# CyberNeura: URL Threat Detection Model Training (URLBERT Tiny v6)
### Google Colab GPU-Accelerated Training Notebook

This notebook fine-tunes **URLBERT Tiny v6** (`CrabInHoney/urlbert-tiny-v6`) on the **PhiUSIIL Phishing URL Dataset** with domain-aware splitting and evaluations.

**Author:** CyberNeura Security AI Team  
**Hardware:** NVIDIA T4 / V100 / A100 GPU

In [ ]:
# 1. Environment Setup & Google Drive Mount
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets evaluate accelerate scikit-learn pandas torch

In [ ]:
# 2. Check GPU Acceleration
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Running on CPU. Switch to GPU runtime via Runtime -> Change runtime type.')

In [ ]:
# 3. Load & Preprocess PhiUSIIL Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Path to dataset in Google Drive or local workspace
DATASET_PATH = Path('/content/drive/MyDrive/CyberNeura/backend/ml/datasets/PhiUSIIL_Phishing_URL_Dataset.csv')
if not DATASET_PATH.exists():
    DATASET_PATH = Path('PhiUSIIL_Phishing_URL_Dataset.csv')

print(f'Loading dataset from {DATASET_PATH}...')
df = pd.read_csv(DATASET_PATH, low_memory=False)
col_map = {c: c.strip().lower() for c in df.columns}
df = df.rename(columns=col_map)

url_col = 'url' if 'url' in df.columns else df.columns[1]
label_col = 'label' if 'label' in df.columns else df.columns[-1]
df = df[[url_col, label_col]].dropna().copy()
df.columns = ['url', 'raw_label']
df['url'] = df['url'].astype(str).str.strip()

# Normalize labels: 0 = LEGITIMATE, 1 = PHISHING
# In raw PhiUSIIL: 1 = legitimate, 0 = phishing
df['label'] = df['raw_label'].apply(lambda v: 0 if int(v) == 1 else 1)
df = df.drop_duplicates(subset=['url']).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.10, stratify=train_df['label'], random_state=42)
print(f'Dataset Split: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}')

In [ ]:
# 4. Tokenizer & Dataset Tokenization (URLBERT Tiny v6)
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_ID = 'CrabInHoney/urlbert-tiny-v6'
print(f'Loading tokenizer for {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(examples):
    return tokenizer(examples['url'], padding='max_length', truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df[['url', 'label']]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df[['url', 'label']]).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['url', 'label']]).map(tokenize_fn, batched=True)

In [ ]:
# 5. Model Initialization & Training Arguments
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: 'LEGITIMATE', 1: 'PHISHING'},
    label2id={'LEGITIMATE': 0, 'PHISHING': 1},
)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels)['f1']
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./cyberneura-urlbert-checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# 6. Fine-Tuning Execution & Checkpointing
print('Starting URLBERT fine-tuning...')
trainer.train()

# 7. Save Final Model Artifacts to Google Drive
SAVE_DIR = '/content/drive/MyDrive/CyberNeura/ml-service/models/url_model'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model and tokenizer saved to {SAVE_DIR}')

In [ ]:
# 8. Test Evaluation on Independent Holdout Set
eval_results = trainer.evaluate(test_ds)
print('Test Set Performance:', eval_results)